# Two-stage soft sensor: predict COD first, then BOD
Data: `ActualData_Subsample.xlsx` (10% public subsample of the essential-oil WRRF dataset, Hassnain et al. 2025).

**Plan**
1. **Stage 1:** predict COD from cheap / online inputs (no COD at deployment).
2. **Stage 2 (Approach B):** predict BOD from the same inputs + *predicted* COD.
3. **Approach A (baseline):** predict BOD directly from the same inputs.
4. Compare A vs B on a **chronological** test set and with **rolling-origin** folds, across several models.

Measured COD / BOD are used only as training targets and for scoring, never as inputs at prediction time.
(One extra reference row, "oracle", feeds the *true* COD into the BOD model. It is not deployable and only shows the best B could ever do.)

Run the cells top to bottom. Only the CONFIG cell needs editing.

In [ ]:
!pip -q install shap xgboost

## 1. Load

In [ ]:
import re, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

from google.colab import files
up = files.upload()                       # upload ActualData_Subsample.xlsx
path = list(up)[0]
raw = pd.read_excel(path)
raw.columns = [re.sub(r"\s+", " ", str(c).strip()) for c in raw.columns]   # "F-SS  (mg/L)" -> "F-SS (mg/L)"
print(raw.shape); print(list(raw.columns))

## 2. CONFIG (edit this cell)
`FEATURES` are the inputs that must be available at prediction time. In this plant `F-SS`, `F-TKN`, `F-NO3` (and `F-COD`, `F-BOD`) come from lab analysis, so they are **not** in the default list. Add `F-SS (mg/L)` only if you would have an online TSS sensor.

In [ ]:
TS   = "Timestamp"
COD  = "F-COD (mg/L)"
BOD  = "F-BOD (mg/L)"
FEATURES = ["F-PH", "Temperature (C)", "DO (mg/L)", "Flow (L/s)",
            "Flow-Recycle (L/s)", "Flow-Aeration (L/s)", "SRT (days)"]
# FEATURES += ["F-SS (mg/L)"]            # uncomment if TSS is available online

TEST_FRAC = 0.15          # last 15% (by time) = held-out test set
N_FOLDS_ROLLING = 6
ROLLING_MODELS = ["Ridge", "SVR", "RandomForest", "ExtraTrees", "HistGB", "XGBoost"]   # add "MLP" if you want (slow, unstable here)
FAST = False              # True = smaller grids and fewer trees: about 5 min instead of 8-10 min (on 2 CPUs)
SEED = 42

## 3. Clean + data checks
Rows are sorted by timestamp (the file is shuffled). The file has several rows per day, so order *within* a day is unknown. No rows have missing values.

In [ ]:
df = raw.copy()
df[TS] = pd.to_datetime(df[TS])
df = df.sort_values(TS, kind="stable").reset_index(drop=True)
df = df[[TS, COD, BOD] + FEATURES].apply(lambda s: pd.to_numeric(s, errors="coerce") if s.name != TS else s).dropna().reset_index(drop=True)

print("rows:", len(df), "|", df[TS].min().date(), "->", df[TS].max().date())
print("timestamps with >1 row:", int(df[TS].duplicated(keep=False).sum()), "rows")
print("share of rows with BOD > COD:", round(float((df[BOD] > df[COD]).mean()), 3),
      " <- BOD5 is normally below COD; mention this as a data-quality note")
display(df[[COD, BOD] + FEATURES].describe().T[["mean", "std", "min", "max"]].round(2))
display(df[[COD, BOD] + FEATURES].corr().round(2)[[COD, BOD]])

## 4. Split, metrics, model zoo
- Train = first 85% of the timeline, test = last 15%. Hyperparameters are tuned **only on train** with `TimeSeriesSplit`.
- **R2 here is sklearn's `r2_score`, which equals NSE.**
- Stage-2 training uses *out-of-fold* COD predictions (5 contiguous blocks, no shuffling), not the true COD. This keeps the training input as noisy as what the model will see at test time.
- "Best model" is chosen by the tuning CV score, not by the test score.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit, KFold, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None

X_all = df[FEATURES].values
y_cod = df[COD].values
y_bod = df[BOD].values
n = len(df); split = int(n * (1 - TEST_FRAC))
Xtr, Xte = X_all[:split], X_all[split:]
ctr, cte = y_cod[:split], y_cod[split:]
btr, bte = y_bod[:split], y_bod[split:]
print(f"train {split} rows ({df[TS].iloc[0].date()} -> {df[TS].iloc[split-1].date()}) | "
      f"test {n-split} rows ({df[TS].iloc[split].date()} -> {df[TS].iloc[-1].date()})")

def score(y, p):
    return {"R2": r2_score(y, p), "RMSE": float(np.sqrt(mean_squared_error(y, p))), "MAE": mean_absolute_error(y, p)}

def ty(est):   # scale the target too (needed for SVR / MLP when targets are in the hundreds)
    return TransformedTargetRegressor(regressor=est, transformer=StandardScaler())

def zoo():
    nt = 100 if FAST else 300
    tg = {"max_depth": [None], "min_samples_leaf": [1, 5]} if FAST else {"max_depth": [8, None], "min_samples_leaf": [1, 5]}
    z = {
        "Ridge": (make_pipeline(StandardScaler(), Ridge()), {"ridge__alpha": [0.1, 1, 10, 100]}),
        "SVR": (ty(make_pipeline(StandardScaler(), SVR())),
                {"regressor__svr__C": [1, 10, 100], "regressor__svr__epsilon": [0.01, 0.1],
                 "regressor__svr__gamma": ["scale", 0.05]}),
        "MLP": (ty(make_pipeline(StandardScaler(), MLPRegressor(max_iter=2000, early_stopping=True, random_state=SEED))),
                {"regressor__mlpregressor__hidden_layer_sizes": [(32,), (64, 32)],
                 "regressor__mlpregressor__alpha": [1e-3]}),
        "RandomForest": (RandomForestRegressor(n_estimators=nt, random_state=SEED, n_jobs=1), tg),
        "ExtraTrees": (ExtraTreesRegressor(n_estimators=nt, random_state=SEED, n_jobs=1), tg),
        "HistGB": (HistGradientBoostingRegressor(random_state=SEED),
                   {"learning_rate": [0.05, 0.1], "max_depth": [3, None], "min_samples_leaf": [20, 50]} if not FAST
                   else {"learning_rate": [0.1], "max_depth": [3, None], "min_samples_leaf": [20]}),
    }
    if XGBRegressor is not None:
        z["XGBoost"] = (XGBRegressor(n_estimators=nt, random_state=SEED, n_jobs=1, verbosity=0),
                        {"max_depth": [3, 6], "learning_rate": [0.05, 0.1]} if not FAST else {"max_depth": [3, 6], "learning_rate": [0.1]})
    return z

MODELS = list(zoo())
print("models:", MODELS)

def tune(name, X, y):
    est, grid = zoo()[name]
    gs = GridSearchCV(est, grid, cv=TimeSeriesSplit(5), scoring="neg_root_mean_squared_error", n_jobs=-1).fit(X, y)
    return gs.best_estimator_, -gs.best_score_

def oof_predict(est, X, y, k=5):
    out = np.zeros(len(y))
    for a, b in KFold(k).split(X):                 # contiguous blocks, no shuffling
        out[b] = clone(est).fit(X[a], y[a]).predict(X[b])
    return out

## 5. Stage 1: predict COD

In [ ]:
t0 = time.time()
stage1 = {}
for m in MODELS:
    est, cv_rmse = tune(m, Xtr, ctr)
    p = est.predict(Xte)
    stage1[m] = {"est": est, "cv_rmse": cv_rmse, "test_pred": p, "oof_train": oof_predict(est, Xtr, ctr)}
cod_tab = pd.DataFrame({m: {**score(cte, stage1[m]["test_pred"]), "CV_RMSE(train)": stage1[m]["cv_rmse"]} for m in MODELS}).T
cod_tab = cod_tab.sort_values("CV_RMSE(train)").round(3)
print(f"COD model scores on the test set  ({time.time()-t0:.0f}s). Sorted by tuning-CV RMSE.")
display(cod_tab)

## 6. Approach A (direct BOD) vs Approach B (two-stage) vs oracle
- **A:** BOD from `FEATURES`.
- **B:** BOD from `FEATURES` + predicted COD (same model family used for both stages).
- **Oracle:** BOD from `FEATURES` + *true* COD (not deployable, upper bound for B).

In [ ]:
t0 = time.time()
res, A_models, B_models, preds = [], {}, {}, {}
for m in MODELS:
    estA, cvA = tune(m, Xtr, btr)
    pA = estA.predict(Xte)

    Xtr_B = np.c_[Xtr, stage1[m]["oof_train"]]
    Xte_B = np.c_[Xte, stage1[m]["test_pred"]]
    estB, cvB = tune(m, Xtr_B, btr)
    pB = estB.predict(Xte_B)

    estO = clone(estA).fit(np.c_[Xtr, ctr], btr)             # oracle reuses A's tuned settings (reference only)
    cvO, pO = np.nan, estO.predict(np.c_[Xte, cte])

    A_models[m], B_models[m] = estA, estB
    preds[m] = {"A": pA, "B": pB, "Oracle": pO}
    for tag, p, cv in [("A direct", pA, cvA), ("B two-stage", pB, cvB), ("Oracle (true COD)", pO, cvO)]:
        res.append({"model": m, "approach": tag, **score(bte, p), "CV_RMSE(train)": cv})
res = pd.DataFrame(res)
print(f"BOD results on the test set ({time.time()-t0:.0f}s)")
tab = res.pivot(index="model", columns="approach", values="R2").round(3)
tab["B - A"] = (tab["B two-stage"] - tab["A direct"]).round(3)
display(tab.sort_values("A direct", ascending=False))
print("\nFull table (R2 = NSE):")
display(res.round(3).set_index(["model", "approach"]))

## 7. Plots

In [ ]:
bestA = res[res.approach == "A direct"].sort_values("CV_RMSE(train)").iloc[0]["model"]
bestB = res[res.approach == "B two-stage"].sort_values("CV_RMSE(train)").iloc[0]["model"]
bestC = cod_tab.index[0]
print("best (by tuning CV): COD ->", bestC, "| BOD A ->", bestA, "| BOD B ->", bestB)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
ax = axes[0]; p = stage1[bestC]["test_pred"]
ax.scatter(cte, p, s=8, alpha=.5); lo, hi = min(cte.min(), p.min()), max(cte.max(), p.max()); ax.plot([lo, hi], [lo, hi], "k--", lw=1)
ax.set_title(f"COD, {bestC}  R2={r2_score(cte, p):.2f}"); ax.set_xlabel("Observed COD (mg/L)"); ax.set_ylabel("Predicted")
ax = axes[1]
for tag, mm, key in [("A direct", bestA, "A"), ("B two-stage", bestB, "B")]:
    p = preds[mm][key]; ax.scatter(bte, p, s=8, alpha=.45, label=f"{tag} ({mm}) R2={r2_score(bte, p):.2f}")
lo, hi = bte.min(), bte.max(); ax.plot([lo, hi], [lo, hi], "k--", lw=1); ax.legend(fontsize=7)
ax.set_title("BOD, test set"); ax.set_xlabel("Observed BOD (mg/L)"); ax.set_ylabel("Predicted")
ax = axes[2]; x = np.arange(len(MODELS)); w = 0.27
for j, (tag, col) in enumerate([("A direct", "A direct"), ("B two-stage", "B two-stage"), ("Oracle", "Oracle (true COD)")]):
    ax.bar(x + (j - 1) * w, [tab.loc[m, col] for m in MODELS], w, label=tag)
ax.set_xticks(x); ax.set_xticklabels(MODELS, rotation=30, ha="right"); ax.set_ylabel("BOD R2 (test)"); ax.legend(fontsize=7)
ax.set_title("A vs B per model")
plt.tight_layout(); plt.show()

plt.figure(figsize=(11, 3.4)); plt.plot(bte, lw=1, label="Observed")
plt.plot(preds[bestA]["A"], lw=1, label=f"A: {bestA}"); plt.plot(preds[bestB]["B"], lw=1, label=f"B: {bestB}")
plt.xlabel("Test sample (time order)"); plt.ylabel("BOD (mg/L)"); plt.legend(); plt.tight_layout(); plt.show()

## 8. Rolling-origin check (is A vs B stable over time?)
Expanding training window, six consecutive test blocks, hyperparameters fixed from section 6/5. For each fold the stage-1 model is re-fit and the stage-2 model sees out-of-fold COD predictions.

In [ ]:
t0 = time.time()
edges = np.linspace(int(n * 0.4), n, N_FOLDS_ROLLING + 1).astype(int)
rows = []
for m in [r for r in ROLLING_MODELS if r in MODELS]:
    for f in range(N_FOLDS_ROLLING):
        a, b = edges[f], edges[f + 1]
        X1, X2, c1, b1, b2 = X_all[:a], X_all[a:b], y_cod[:a], y_bod[:a], y_bod[a:b]
        pA = clone(A_models[m]).fit(X1, b1).predict(X2)
        e1 = clone(stage1[m]["est"]); oof = oof_predict(e1, X1, c1); c_hat = e1.fit(X1, c1).predict(X2)
        pB = clone(B_models[m]).fit(np.c_[X1, oof], b1).predict(np.c_[X2, c_hat])
        rows.append({"model": m, "fold": f + 1, "A": r2_score(b2, pA), "B": r2_score(b2, pB)})
roll = pd.DataFrame(rows)
summ = roll.groupby("model").agg(A_mean=("A", "mean"), A_std=("A", "std"), B_mean=("B", "mean"), B_std=("B", "std"))
summ["B-A mean"] = summ["B_mean"] - summ["A_mean"]
summ["folds where B > A"] = roll.assign(w=roll.B > roll.A).groupby("model").w.sum()
print(f"Rolling-origin R2 over {N_FOLDS_ROLLING} folds ({time.time()-t0:.0f}s)")
display(summ.round(3).sort_values("A_mean", ascending=False))

## 9. Which inputs matter? (permutation importance + SHAP)

In [ ]:
from sklearn.inspection import permutation_importance
mdl = A_models[bestA]
pi = permutation_importance(mdl, Xte, bte, n_repeats=20, random_state=SEED, scoring="r2")
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values()
imp.plot.barh(figsize=(6, 3.5), title=f"Permutation importance on test (BOD, A, {bestA})\n(drop in R2 when the input is shuffled)")
plt.tight_layout(); plt.show()

try:
    import shap
    tree_name = next(t for t in ["ExtraTrees", "RandomForest", "XGBoost", "HistGB"] if t in A_models)
    tm = A_models[tree_name]
    sv = shap.Explainer(tm, Xtr[:500])(Xte[:400]) if tree_name == "HistGB" else shap.TreeExplainer(tm)(Xte[:400])
    sv.feature_names = FEATURES
    shap.plots.beeswarm(sv, show=True)
except Exception as e:
    print("SHAP skipped:", e)

## Notes for the write-up
- **How to read A vs B:** if B is not clearly above A across models *and* across rolling folds, the extra COD stage is not justified for this dataset. The oracle row bounds how much B could gain.
- **Chronological evaluation** is stricter than the shuffled CV used in the source paper (which reports R2 around 0.98). Expect lower numbers here, and say why.
- The MLP's stage-2 results swing a lot between folds; do not read much into them.
- This is a 10% subsample, so rows are thinner in time than the real 8-samples-a-day record.
- BOD > COD in most rows is unusual; the paper does not explain it. State it as a data-quality limitation.
- Permutation importance with correlated inputs can understate each input; do not treat the ranking as causal.
- This dataset is an industrial essential-oil plant in Australia. It shows the method works on real industrial data, not that it transfers to Lake Qarun without local data.